<a href="https://colab.research.google.com/github/Sanjana-Ydv/Marketing_performance_dashboard/blob/main/Marketing_Performance_Major_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Name = SANJANA YADAV**

**Batch = Data Analysis August 2026**


In [ ]:
# 1. DATA CLEANING

In [ ]:
import pandas as pd
df = pd.read_csv("blinkit_marketing_performance.csv")
print("Shape:", df.shape)
print(df.info())

Shape: (5400, 11)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5400 entries, 0 to 5399
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   campaign_id        5400 non-null   int64  
 1   campaign_name      5400 non-null   object 
 2   date               5400 non-null   object 
 3   target_audience    5400 non-null   object 
 4   channel            5400 non-null   object 
 5   impressions        5400 non-null   int64  
 6   clicks             5400 non-null   int64  
 7   conversions        5400 non-null   int64  
 8   spend              5400 non-null   float64
 9   revenue_generated  5400 non-null   float64
 10  roas               5400 non-null   float64
dtypes: float64(3), int64(4), object(4)
memory usage: 464.2+ KB
None


In [ ]:
df = df.rename(columns={"revenue_generated": "revenue", "roas": "roas_original"})
df["date"] = pd.to_datetime(df["date"])
for column in ["campaign_name", "target_audience", "channel"]:
    df[column] = df[column].str.strip()

print("Missing_values", df.isnull().sum())
df = df.drop_duplicates()

Missing_values campaign_id        0
campaign_name      0
date               0
target_audience    0
channel            0
impressions        0
clicks             0
conversions        0
spend              0
revenue            0
roas_original      0
dtype: int64


In [ ]:
checks = {
    "clicks > impressions": (df["clicks"] > df["impressions"]).sum(),
    "conversions > clicks": (df["conversions"] > df["clicks"]).sum(),
    "negative spend":       (df["spend"] < 0).sum(),
    "zero spend":           (df["spend"] == 0).sum(),
}
print("Validation checks (0 = good):", checks)


Validation checks (0 = good): {'clicks > impressions': np.int64(0), 'conversions > clicks': np.int64(0), 'negative spend': np.int64(0), 'zero spend': np.int64(0)}


In [ ]:
df["ctr"]             = df["clicks"] / df["impressions"] * 100
df["conversion_rate"] = df["conversions"] / df["clicks"] * 100
df["cpc"]             = df["spend"] / df["clicks"]
df["cpa"]             = df["spend"] / df["conversions"]
df["roas"]            = df["revenue"] / df["spend"]
df["roi"]             = (df["revenue"] - df["spend"]) / df["spend"] * 100

mismatch = ((df["roas"] - df["roas_original"]).abs() > 0.01).sum()
print(f"Rows where original roas != revenue/spend: {mismatch} of {len(df)}")

df["month"] = df["date"].dt.to_period("M").astype(str)

def summarize(group_col):
      g = df.groupby(group_col).agg(
          impressions=("impressions", "sum"),
          clicks=("clicks", "sum"),
          conversions=("conversions", "sum"),
          spend=("spend", "sum"),
          revenue=("revenue", "sum"),
      )
      g["ctr"] = g["clicks"] / g["impressions"] * 100
      g["conversion_rate"] = g["conversions"] / g["clicks"] * 100
      g["roi"] = (g["revenue"] - g["spend"]) / g["spend"] * 100
      return g.round(2).sort_values("roi", ascending=False)

print("By channel:", summarize("channel"))
print("By campaign:", summarize("campaign_name"))
print("By audience:", summarize("target_audience"))
print("Monthly trend:", summarize("month").sort_index())


df.to_csv("cleaned_data.csv", index=False)
print("Saved cleaned_data.csv")


Rows where original roas != revenue/spend: 5374 of 5400
By channel:               impressions  clicks  conversions       spend     revenue    ctr  \
channel                                                                         
Email             7275694  756057        74671  3997488.04  8189331.58  10.39   
SMS               7367010  744274        73235  3998607.54  7938649.32  10.10   
Social Media      7281244  738839        74940  4110363.91  7990415.98  10.15   
App               7563662  734975        75192  4213378.75  8075010.49   9.72   

              conversion_rate     roi  
channel                                
Email                    9.88  104.86  
SMS                      9.84   98.54  
Social Media            10.14   94.40  
App                     10.23   91.65  
By campaign:                        impressions  clicks  conversions       spend  \
campaign_name                                                         
Referral Program           3276426  329302        